# Porownanie architektur pod domain shift (RarePlanes) — Colab

Trening na **podzbiorze 10k synthetic** (stratyfikowany po 15 lotniskach), ewaluacja cross-domain na **realnym holdoucie**.

Architektury: **RT-DETR-l** (stabilny: lr=1e-4 + cosine) vs YOLOv10n (ref lokalny: mAP@50 0.459). Opcjonalnie D-FINE.

**Wymagania:** GPU runtime (A100). Dane ~31 GB (30 GB synth 10k + 0.9 GB real test).

⚠️ Colab ulotny — wynik pobierany automatycznie na koncu (komorka 6).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# 1. Repo + zaleznosci
!git clone https://github.com/JakubPaszke/rareplanes-domain-shift.git
%cd rareplanes-domain-shift
!pip -q install ultralytics pycocotools

In [ ]:
# 2. Adnotacje + realny test (tarball)
import os
for d in ['data/real/annotations','data/real/tarballs','data/synthetic/annotations','data/synthetic/images/train']:
    os.makedirs(d, exist_ok=True)
B='https://rareplanes-public.s3.amazonaws.com'
for f in ['instances_train_aircraft.json','instances_test_aircraft.json']:
    !curl -sf -o data/real/annotations/{f} {B}/real/metadata_annotations/{f}
    !curl -sf -o data/synthetic/annotations/{f} {B}/synthetic/metadata_annotations/{f}
!curl -sf -o data/real/tarballs/test.tar.gz {B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz
!mkdir -p data/real/PS-RGB_tiled && tar -xzf data/real/tarballs/test.tar.gz -C data/real/PS-RGB_tiled/
print('real test tiles:', len(os.listdir('data/real/PS-RGB_tiled/PS-RGB_tiled')))

In [ ]:
# 3. Synthetic 10k — pobieranie czystym Pythonem (niezawodne; 32 watki, resume)
import urllib.request
from concurrent.futures import ThreadPoolExecutor
DST='data/synthetic/images/train'
files=[l.strip() for l in open('configs/synthetic_10k_train_files.txt')]
files+=[l.strip() for l in open('configs/synthetic_10k_val_files.txt')]
files=[f for f in files if f]
print('do pobrania:', len(files))
def fetch(fn):
    dst=f'{DST}/{fn}'
    if os.path.exists(dst) and os.path.getsize(dst)>0: return True
    try:
        urllib.request.urlretrieve(f'{B}/synthetic/train/images/{fn}', dst+'.part')
        os.rename(dst+'.part', dst); return True
    except Exception:
        if os.path.exists(dst+'.part'): os.remove(dst+'.part')
        return False
ok=0
with ThreadPoolExecutor(max_workers=32) as ex:
    for i,r in enumerate(ex.map(fetch, files),1):
        ok+=r
        if i%2000==0: print(f'  {i}/{len(files)} (ok={ok})')
have=len([f for f in os.listdir(DST) if f.endswith('.png')])
print(f'GOTOWE: pobrano {ok}/{len(files)}, na dysku {have} png')
assert have > len(files)*0.99, 'ZA MALO OBRAZOW — uruchom komorke ponownie (resume dociagnie reszte)'

In [ ]:
# 4. Konwersja COCO->YOLO
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15
import os
n=len(os.listdir('data/yolo/synthetic_aircraft/images/train'))
print('YOLO train images:', n); assert n>9000, 'konwersja nie zlinkowala obrazow!'

In [ ]:
# 5. RT-DETR-l — STABILNY trening (lr=1e-4 + cosine + AdamW + warmup 5)
#    bez tego: auto-lr 0.002 -> rozbieznosc do nan po ~20 epokach
!python3 src/train_yolo.py --data data/yolo/synthetic_aircraft/data.yaml \
  --name rtdetr_l_10k --model rtdetr-l.pt \
  --epochs 60 --batch 16 --imgsz 512 --seed 42 --patience 20 --workers 8 \
  --optimizer AdamW --lr0 0.0001 --cos_lr --warmup_epochs 5

In [ ]:
# 6. Ewaluacja cross-domain na REALNYM tescie + AUTO-DOWNLOAD wyniku
!python3 src/eval_per_size.py --weights runs/rtdetr_l_10k/weights/best.pt \
  --img-dir data/real/PS-RGB_tiled/PS-RGB_tiled \
  --coco-gt data/real/annotations/instances_test_aircraft.json --name rtdetr_l_10k_ml
import json
print(json.dumps(json.load(open('results/per_size/rtdetr_l_10k_ml.json'))['metrics'], indent=2))
from google.colab import files
files.download('results/per_size/rtdetr_l_10k_ml.json')
files.download('runs/rtdetr_l_10k/weights/best.pt')

## D-FINE (opcjonalnie, trzecia architektura)
Zmien `--model` na wage D-FINE i powtorz komorki 5-6 z `--name dfine_10k`.

## Sanity-check stabilnosci
Jesli w logach treningu pojawi sie `nan` lub mAP spadnie do ~0 — lr nadal za wysoki: zmniejsz `--lr0` do 5e-5.
Oczekiwane: mAP_syn rosnie monotonicznie do ~0.95+ bez zalaman.

Punkty odniesienia (10k, real test): YOLOv10n mAP@50 **0.459**; RT-DETR (ep.2, niestabilny) 0.489.